In [ ]:
# 1. setup
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATASET_DIR = os.path.join('..', 'dataset')
OUTPUT_DIR  = os.path.join('..', 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'kaplan_meier'), exist_ok=True)

print('ready')

In [ ]:
# 2. load TNBC
tcga_brca = pd.read_csv(os.path.join(DATASET_DIR, 'brca_tcga_clinical_data.tsv'), sep='\t', low_memory=False)

er_col   = 'ER Status By IHC'
pr_col   = 'PR status by ihc'
her2_col = 'IHC-HER2'

tnbc = tcga_brca[
    (tcga_brca[er_col]   == 'Negative') &
    (tcga_brca[pr_col]   == 'Negative') &
    (tcga_brca[her2_col] == 'Negative')
].copy().reset_index(drop=True)

print(f'TNBC: {len(tnbc)}')
tnbc[['Patient ID', er_col, pr_col, her2_col]].head()

In [ ]:
# 3. build scores
import tarfile
from bioconverge.utils import parse_gmt

RNASEQ_TAR = os.path.join(DATASET_DIR, 'gdac.broadinstitute.org_BRCA.Merge_rnaseqv2__illuminahiseq_rnaseqv2__unc_edu__Level_3__RSEM_genes_normalized__data.Level_3.2016012800.0.0.tar.gz')
MAF_TAR    = os.path.join(DATASET_DIR, 'gdac.broadinstitute.org_BRCA.Mutation_Packager_Oncotated_Calls.Level_3.2016012800.0.0.tar.gz')
GMT_PATH   = os.path.join(DATASET_DIR, 'h.all.v2023.2.Hs.symbols.gmt')

hallmarks    = parse_gmt(GMT_PATH)
IMMUNE_GENES = ['CD8A', 'CD8B', 'GZMA', 'PRF1']
PROLIF_GENES = ['MKI67']
EMT_GENES    = hallmarks.get('HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION', [])
print(f'EMT genes: {len(EMT_GENES)}')

def safe_norm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else pd.Series(0.5, index=s.index)

def load_rnaseq(tar_path, patient_ids, gene_sets):
    with tarfile.open(tar_path) as t:
        member = next(m for m in t.getmembers() if m.name.endswith('.data.txt'))
        raw = pd.read_csv(t.extractfile(member), sep='\t', header=0, index_col=0, skiprows=[1])
    raw.columns = [c[:12] for c in raw.columns]
    raw.index   = [g.split('|')[0] for g in raw.index]
    raw = raw[~raw.index.duplicated(keep='first')]
    pid_set = set(patient_ids)
    common_cols = list(dict.fromkeys(c for c in raw.columns if c in pid_set))
    rna = raw[common_cols].T
    rna.index.name = 'patient_id'
    rna = rna.groupby('patient_id').mean().T
    unique_pids = list(dict.fromkeys(p for p in patient_ids if p in rna.columns))
    rna = rna[unique_pids]
    scores = {}
    for name, genes in gene_sets.items():
        avail = [g for g in genes if g in rna.index]
        scores[name] = rna.loc[avail].mean(axis=0) if avail else pd.Series(np.nan, index=unique_pids)
    df = pd.DataFrame(scores)
    df.index.name = 'patient_id'
    return df.reset_index(), unique_pids

def load_tmb(tar_path, patient_ids):
    pid_set = set(patient_ids)
    counts = {p: 0 for p in pid_set}
    with tarfile.open(tar_path) as t:
        for member in t.getmembers():
            if not member.name.endswith('.maf.txt'):
                continue
            pid = os.path.basename(member.name)[:12]
            if pid not in pid_set:
                continue
            try:
                df = pd.read_csv(t.extractfile(member), sep='\t', comment='#',
                                 usecols=['Hugo_Symbol'], low_memory=False)
                counts[pid] = len(df)
            except Exception:
                pass
    return pd.Series(counts, name='mutation_raw')

patient_ids = list(dict.fromkeys(tnbc['Patient ID'].tolist()))
print(f'patients: {len(patient_ids)}')

rna_df = None
rna_matched = []
if os.path.exists(RNASEQ_TAR):
    print('loading RNA-seq')
    rna_df, rna_matched = load_rnaseq(RNASEQ_TAR, patient_ids,
                                       {'prolif_raw': PROLIF_GENES,
                                        'immune_raw': IMMUNE_GENES,
                                        'emt_raw':    EMT_GENES})
    print(f'RNA-seq matched: {len(rna_matched)}')
else:
    print('RNA-seq not found, using proxies')

print('loading mutations')
tmb = load_tmb(MAF_TAR, patient_ids)
print(f'MAF matched: {(tmb > 0).sum()}')

clin = tnbc[['Patient ID', 'Fraction Genome Altered', 'Longest Dimension']].copy()
clin.columns = ['patient_id', 'fga', 'longest_dim']
clin['fga']         = pd.to_numeric(clin['fga'], errors='coerce')
clin['longest_dim'] = pd.to_numeric(clin['longest_dim'], errors='coerce')
clin = clin.drop_duplicates(subset='patient_id').reset_index(drop=True)

if rna_df is not None:
    score_df = clin.merge(rna_df, on='patient_id', how='left')
    score_df['proliferation_score'] = safe_norm(np.log1p(score_df['prolif_raw'].fillna(score_df['prolif_raw'].median())))
    score_df['immune_score']        = safe_norm(np.log1p(score_df['immune_raw'].fillna(score_df['immune_raw'].median())))
    score_df['emt_score']           = safe_norm(np.log1p(score_df['emt_raw'].fillna(score_df['emt_raw'].median())))
else:
    score_df = clin.copy()
    rng = np.random.default_rng(42)
    score_df['proliferation_score'] = safe_norm(pd.Series(rng.uniform(0, 1, len(score_df))))
    score_df['immune_score']        = safe_norm(pd.Series(rng.uniform(0, 1, len(score_df))))
    score_df['emt_score']           = safe_norm(pd.Series(rng.uniform(0, 1, len(score_df))))

score_df['mutation_raw']   = score_df['patient_id'].map(tmb.to_dict()).fillna(0)
score_df['genomic_score']  = safe_norm(score_df['fga'].fillna(score_df['fga'].median()))
score_df['size_score']     = safe_norm(score_df['longest_dim'].fillna(score_df['longest_dim'].median()))
score_df['mutation_score'] = safe_norm(np.log1p(score_df['mutation_raw']))

score_df = score_df[['patient_id', 'genomic_score', 'size_score',
                      'proliferation_score', 'immune_score', 'mutation_score', 'emt_score']].dropna().reset_index(drop=True)

score_df.to_csv(os.path.join(OUTPUT_DIR, 'score_matrix.csv'), index=False)
print(f'score matrix: {score_df.shape}')
score_df.head()

In [ ]:
# 4. layer1 concordance
from bioconverge.layer1 import ConcordanceAnalyzer

l1 = ConcordanceAnalyzer(score_df, 'patient_id')
l1.fit(n_archetypes=3, n_bootstrap=500)

conc = l1.concordance()
conv = l1.convergence()
arch = l1.archetypes()
stab = l1.stability()

conc.to_csv(os.path.join(OUTPUT_DIR, 'concordance.csv'), index=False)
conv.to_csv(os.path.join(OUTPUT_DIR, 'convergence.csv'), index=False)
arch.to_csv(os.path.join(OUTPUT_DIR, 'archetypes.csv'), index=False)

print(conc.to_string())
print(f'stability: {stab}')

In [ ]:
# 5. concordance plot
score_names = sorted(set(conc['score_a'].tolist() + conc['score_b'].tolist()))
n = len(score_names)
mat = np.zeros((n, n))
idx_map = {s: i for i, s in enumerate(score_names)}
for _, row in conc.iterrows():
    i, j = idx_map[row['score_a']], idx_map[row['score_b']]
    v = row['spearman_rho'] if not np.isnan(row['spearman_rho']) else 0
    mat[i, j] = v
    mat[j, i] = v
np.fill_diagonal(mat, 1.0)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(mat, cmap='RdBu', vmin=-1, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(score_names, rotation=45, ha='right')
ax.set_yticklabels(score_names)
plt.colorbar(im, ax=ax, label='spearman rho')
ax.set_title('concordance matrix')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'concordance_matrix.png'), dpi=150)
plt.show()
print('saved')

In [ ]:
# 6. convergence plot
colors = {'convergent_high': '#2ecc71', 'convergent_mid': '#f39c12', 'convergent_low': '#e74c3c', 'unknown': '#95a5a6'}
fig, ax = plt.subplots(figsize=(8, 4))
for stratum, grp in conv.groupby('stratum'):
    ax.hist(grp['convergence_index'], bins=15, alpha=0.7, label=stratum, color=colors.get(stratum, 'gray'))
ax.set_xlabel('convergence index')
ax.set_ylabel('count')
ax.set_title('patient convergence strata')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'convergence_strata.png'), dpi=150)
plt.show()
print(conv['stratum'].value_counts().to_string())

In [ ]:
# 7. archetype summary
arch_summary = arch.merge(conv, on='patient_id')
print(arch_summary.groupby('archetype')['stratum'].value_counts().unstack(fill_value=0).to_string())
disc = l1.discordance()
print(f'discordant: {len(disc)}')

In [ ]:
# 8. score metadata
from bioconverge.layer3 import HypothesisGenerator
from bioconverge.layer4 import LEHMANN_SIGNATURES

score_metadata = {
    'genomic_score':       {'process': 'DNA repair instability',                    'modality': 'genomic'},
    'size_score':          {'process': 'tumor growth and growth factor signaling',   'modality': 'imaging'},
    'proliferation_score': {'process': 'cell cycle proliferation',                  'modality': 'transcriptomic'},
    'immune_score':        {'process': 'immune activation',                         'modality': 'transcriptomic'},
    'mutation_score':      {'process': 'mutational burden',                         'modality': 'genomic'},
    'emt_score':           {'process': 'epithelial-mesenchymal transition EMT',      'modality': 'transcriptomic'},
}

for subtype, keywords in LEHMANN_SIGNATURES.items():
    print(f'{subtype}: {keywords}')
print()
for sc, meta in score_metadata.items():
    proc = meta['process']
    hits = [s for s, kws in LEHMANN_SIGNATURES.items() if any(kw.lower() in proc.lower() for kw in kws)]
    print(f'{sc}: {proc!r} -> {hits if hits else "no match"}')

In [ ]:
# 9. hypothesis generation
l3 = HypothesisGenerator(score_metadata, arch)
l3.generate()

hyp = l3.hypotheses()
hyp.to_csv(os.path.join(OUTPUT_DIR, 'hypotheses_raw.csv'), index=False)
print(f'hypotheses: {len(hyp)}')
hyp[['archetype', 'process', 'db_support', 'reactome_pathway', 'pubmed_count', 'pubmed_flag']].head(10)

In [ ]:
# 10. cross support
cs = l3.cross_support()
cs.to_csv(os.path.join(OUTPUT_DIR, 'cross_support.csv'), index=False)
print(cs.to_string())

In [ ]:
# 11. find validation files
from bioconverge.layer4 import ValidationEngine

clinical_tar       = None
mut_tar            = None
metabric_path      = None
tcga_brca_path_val = None

for fname in os.listdir(DATASET_DIR):
    fpath = os.path.join(DATASET_DIR, fname)
    if 'clinical' in fname and fname.endswith('.tar.gz'):
        clinical_tar = fpath
    elif fname == 'clinical.tsv':
        clinical_tar = fpath
    elif 'Mutation_Packager' in fname and fname.endswith('.tar.gz'):
        mut_tar = fpath
    elif 'metabric' in fname and fname.endswith('.tsv'):
        metabric_path = fpath
    elif 'brca_tcga' in fname and fname.endswith('.tsv'):
        tcga_brca_path_val = fpath

print(f'clinical: {clinical_tar}')
print(f'mutations: {mut_tar}')
print(f'metabric: {metabric_path}')
print(f'tcga brca: {tcga_brca_path_val}')

In [ ]:
# 12. run validation
l4 = ValidationEngine(
    clinical_tar_path=clinical_tar,
    mut_tar_path=mut_tar,
    metabric_path=metabric_path,
    tcga_brca_path=tcga_brca_path_val,
    dataset_dir=DATASET_DIR,
)

km_dir = os.path.join(OUTPUT_DIR, 'kaplan_meier')
l4.validate(
    hypotheses_df=hyp,
    archetypes_df=arch,
    score_df=score_df,
    patient_col='patient_id',
    score_metadata=score_metadata,
    km_output_dir=km_dir,
)

replication_rate = l4.compute_replication_rate(hyp)
print(f'replication rate: {replication_rate}')

In [ ]:
# 13. survival results
surv = l4.survival_results()
if surv and not surv.get('skipped'):
    print(f'matched: {surv["n_matched"]}')
    surv['results'].to_csv(os.path.join(OUTPUT_DIR, 'survival_results.csv'), index=False)
    print(surv['results'].to_string())
    for fname in os.listdir(km_dir):
        if fname.endswith('.png'):
            img = plt.imread(os.path.join(km_dir, fname))
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(fname.replace('.png', ''))
            plt.tight_layout()
            plt.show()
else:
    print(f'survival skipped: {surv}')

In [ ]:
# 14. benchmark results
bench = l4.benchmark_results()
if bench and not bench.get('skipped'):
    print(f'mean precision: {bench["mean_precision"]:.3f}')
    print(f'mean recall: {bench["mean_recall"]:.3f}')
    for subtype, vals in bench['lehmann_results'].items():
        print(f'{subtype}: precision={vals["precision"]:.2f}, recall={vals["recall"]:.2f}')
else:
    print(f'benchmark skipped: {bench}')

In [ ]:
# 15. tier breakdown
tiered = l4.tiered_hypotheses()
tiered.to_csv(os.path.join(OUTPUT_DIR, 'hypotheses_tiered.csv'), index=False)

print(tiered['confidence_tier'].value_counts().to_string())

tier_a = tiered[tiered['confidence_tier'] == 'A']
tier_b = tiered[tiered['confidence_tier'] == 'B']
tier_c = tiered[tiered['confidence_tier'] == 'C']
print(f'Tier A: {len(tier_a)}, Tier B: {len(tier_b)}, Tier C: {len(tier_c)}')

tier_counts = tiered['confidence_tier'].value_counts().reindex(['A', 'B', 'C'], fill_value=0)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(tier_counts.index, tier_counts.values, color=['#27ae60', '#f39c12', '#e74c3c'])
ax.set_xlabel('tier')
ax.set_ylabel('hypotheses')
ax.set_title('tier A/B/C breakdown')
for bar, v in zip(bars, tier_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, str(v), ha='center', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'tier_breakdown.png'), dpi=150)
plt.show()
print('saved')

In [ ]:
# 16. best hypothesis
show_tier = tier_a if len(tier_a) > 0 else tier_b if len(tier_b) > 0 else tier_c
best = show_tier.sort_values(['validation_score', 'db_support'], ascending=False).iloc[0]
for col in ['archetype', 'process', 'modality', 'hypothesis', 'db_support',
            'reactome_pathway', 'enrichr_term', 'pubmed_count', 'pubmed_flag', 'validation_score', 'confidence_tier']:
    if col in best.index:
        print(f'{col}: {best[col]}')

In [ ]:
# 17. discordance analysis
conv_low = conv[conv['stratum'] == 'convergent_low']['patient_id'].tolist()
print(f'discordant: {len(conv_low)}')

arch_merged = arch.merge(conv, on='patient_id')
arch_merged.to_csv(os.path.join(OUTPUT_DIR, 'archetype_convergence.csv'), index=False)
for a in sorted(arch_merged['archetype'].unique()):
    sub = arch_merged[arch_merged['archetype'] == a]
    disc_sub = sub[sub['stratum'] == 'convergent_low']
    print(f'archetype {a}: {len(sub)} patients, {len(disc_sub)} discordant ({100*len(disc_sub)/max(len(sub),1):.0f}%)')

In [ ]:
# 18. full report
from bioconverge.report import BioConvergeResult

result = BioConvergeResult(l1=l1, l2=None, l3=l3, l4=l4, skipped=['layer2'])
result.report(OUTPUT_DIR)

for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        print(f'{f}: {os.path.getsize(fpath)} bytes')

In [ ]:
# 19. summary
print('done')
print(f'patients: {len(score_df)}')
print(f'archetypes: {len(arch["archetype"].unique())}')
print(f'hypotheses: {len(hyp)}')
print(f'Tier A: {(tiered["confidence_tier"]=="A").sum()}')
print(f'Tier B: {(tiered["confidence_tier"]=="B").sum()}')
print(f'Tier C: {(tiered["confidence_tier"]=="C").sum()}')
print(f'output: {OUTPUT_DIR}')